In [5]:
import h5py as h5 

import matplotlib.pyplot as plt
from matplotlib import cm, colors
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm
from matplotlib import colors, ticker

import numpy as np
import pandas as pd
import os, sys
import traceback

from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

import h5flow 
plt.style.use('../../utils/dune.mplstyle')
from sklearn.cluster import DBSCAN 

# Path to repo root (two directories above notebook)
light_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(light_root)

from light.PMT_analysis_utils import * 

# Path to repo root (two directories above notebook)
repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(repo_root)

from utils.backtracking import get_charge_event_hits, hit_backtracker, get_ancestry # Now this works
from utils.my_ev_display import event_display

In [6]:
def get_distance(inel_cluster):
    print(inel_cluster['x'])
    print(inel_cluster["x"].iloc[0])
    print(inel_cluster["x"].iloc[1])
    dx = inel_cluster["x"] - inel_cluster["x"]
    dy = inel_cluster["y"] - inel_cluster["y"]
    dz = inel_cluster["z"] - inel_cluster["z"]
    dist = np.sqrt(dx**2 + dy**2 + dz**2)
    return dist

In [7]:
def get_capture_like(df):
    # Remove non-clustered hits
    cluster_energy = [] 
    mean_position = []
    inel_cluster_energy = [] 
    df_filtered = df[df["cluster_label"]!=-1]
    unique_clusters = np.unique(df_filtered["id"])
    for icluster in unique_clusters:
        this_cluster = df_filtered[df_filtered['id']==icluster]
        if((len(this_cluster)>=5)):
            x_mean = np.mean(this_cluster['x'])
            y_mean = np.mean(this_cluster['y'])
            z_mean = np.mean(this_cluster['z'])
            mean_position.append([x_mean,y_mean,z_mean])
            cluster_energy.append(np.sum(this_cluster['E']))

        if((len(this_cluster)<=2)):
            inel_cluster_energy.append(np.sum(this_cluster['E']))
    
    return cluster_energy, np.array(mean_position), inel_cluster_energy
    


In [ ]:
input_csv_no_source = "/pscratch/sd/l/lmlepin/cluster_outputs/no_source_32us_window_20260309_070118_clusters.csv"
df_no_source = pd.read_csv(input_csv_no_source)
n_events_no_source = df_no_source[["light_id", "file_id"]].drop_duplicates().shape[0] 
print(f"Number of no source events {n_events_no_source}") 
no_source_E, no_source_positions, no_source_inelastic_E = get_capture_like(df_no_source)


0            0.0::0.0
1            0.0::2.0
2            0.0::2.0
3            0.0::2.0
4            0.0::3.0
             ...     
89332    19.0::1643.0
89333    19.0::1643.0
89334    19.0::1644.0
89335    19.0::1644.0
89336    19.0::1644.0
Length: 89337, dtype: object
Number of no source events 26916


KeyboardInterrupt: 

In [ ]:
input_csv_source = "/pscratch/sd/l/lmlepin/cluster_outputs/source_in_one_trig_32us_window_20260309_070202_clusters.csv"
df_source = pd.read_csv(input_csv_source)

# Keep the same amount of events as in the background
unique_pairs = (
    df_source[["light_id", "file_id"]]
    .drop_duplicates()
    .head(n_events_no_source)
)

# Step 2: keep only rows matching those pairs
df_filtered_source = df_source.merge(unique_pairs, on=["light_id", "file_id"], how="inner")
source_E, source_positions, source_inelastic_E = get_capture_like(df_filtered_source)

In [ ]:
# Common binning
bins = np.linspace(0., 6., 30)

# NumPy histograms
h_source, edges = np.histogram(source_E, bins=bins)
h_nosrc,  _     = np.histogram(no_source_E, bins=bins)

# Subtraction (clip negatives)
h_sub = np.clip(h_source - h_nosrc, 0, None)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    figsize=(7, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
)

# --- Top panel: source vs no-source ---
ax_top.hist(
    edges[:-1],
    bins=edges,
    weights=h_source,
    histtype="step",
    color="red",
    label="source"
)

ax_top.hist(
    edges[:-1],
    bins=edges,
    weights=h_nosrc,
    histtype="step",
    color="blue",
    label="no source"
)

ax_top.set_ylabel("Number of clusters")
ax_top.legend()
ax_top.tick_params(labelbottom=False)

# --- Bottom panel: subtraction ---
ax_bot.hist(
    edges[:-1],
    bins=edges,
    weights=h_sub,
    histtype="step",
    color="red"
)

ax_bot.set_xlabel("Cluster energy [MeV]")
ax_bot.set_ylabel("Subtraction")
ax_bot.legend()

plt.show()


In [ ]:
# --- Plot subtraction ---
plt.figure()

plt.hist(
    edges[:-1],
    bins=edges,
    weights=h_sub,
    histtype="step",
    color="black",
    label="substracted"
)

plt.xlabel("Cluster energy")
plt.ylabel("Number of clusters")
plt.legend()

In [ ]:
# Common binning
bins = np.linspace(0., 2., 30)

# NumPy histograms
h_source, edges = np.histogram(source_inelastic_E, bins=bins)
h_nosrc,  _     = np.histogram(no_source_inelastic_E, bins=bins)

# Subtraction (clip negatives)
h_sub = np.clip(h_source - h_nosrc, 0, None)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    figsize=(7, 6),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
)

# --- Top panel: source vs no-source ---
ax_top.hist(
    edges[:-1],
    bins=edges,
    weights=h_source,
    histtype="step",
    color="red",
    label="source"
)

ax_top.hist(
    edges[:-1],
    bins=edges,
    weights=h_nosrc,
    histtype="step",
    color="blue",
    label="no source"
)

ax_top.set_ylabel("Number of clusters")
ax_top.legend()
ax_top.tick_params(labelbottom=False)

# --- Bottom panel: subtraction ---
ax_bot.hist(
    edges[:-1],
    bins=edges,
    weights=h_sub,
    histtype="step",
    color="red"
)

ax_bot.set_xlabel("Cluster energy [MeV]")
ax_bot.set_ylabel("Subtraction")
ax_bot.legend()

plt.show()


In [ ]:
# Common binning
xbins = np.linspace(-64, 64, 20)
ybins = np.linspace(-64, 64, 20)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(10, 4),
    sharex=True,
    sharey=True
)

# --- Source ---
h1 = ax1.hist2d(
    source_positions[:, 2],  # z
    source_positions[:, 0],  # x
    bins=[xbins, ybins]
)
ax1.set_title("Source")
ax1.set_xlabel("z [cm]")
ax1.set_ylabel("x [cm]")

# --- No source ---
h2 = ax2.hist2d(
    no_source_positions[:, 2],  # z
    no_source_positions[:, 0],  # x
    bins=[xbins, ybins]
)
ax2.set_title("No source")
ax2.set_xlabel("z [cm]")

# Shared colorbar (use source hist for normalization)
#cbar = fig.colorbar(h1[3], ax=[ax1, ax2])
#cbar.set_label("Counts")


plt.tight_layout()
plt.show()


In [ ]:
# Common binning
xbins = np.linspace(-64, 64, 20)
ybins = np.linspace(-64, 64, 20)

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(10, 4),
    sharex=True,
    sharey=True
)

# --- Source ---
h1 = ax1.hist2d(
    source_positions[:, 2],  # z
    source_positions[:, 1],  # x
    bins=[xbins, ybins]
)
ax1.set_title("Source")
ax1.set_xlabel("z [cm]")
ax1.set_ylabel("y [cm]")

# --- No source ---
h2 = ax2.hist2d(
    no_source_positions[:, 2],  # z
    no_source_positions[:, 1],  # x
    bins=[xbins, ybins]
)
ax2.set_title("No source")
ax2.set_xlabel("z [cm]")

# Shared colorbar (use source hist for normalization)
#cbar = fig.colorbar(h1[3], ax=[ax1, ax2])
#cbar.set_label("Counts")


plt.tight_layout()
plt.show()